In [15]:
from pathlib import Path
import sys, importlib
import numpy as np
import pandas as pd

project_path = Path.cwd().resolve()
while not (project_path / "src").exists() and project_path != project_path.parent:
    project_path = project_path.parent
if str(project_path) not in sys.path:
    sys.path.insert(0, str(project_path))

import src.feature_engineering as fe
import src.possession_model as pm
import src.temporal_analysis as ta
import src.prediction as pr
for module in (fe, pm, ta, pr):
    importlib.reload(module)

paths = pr.default_paths(project_path)
print("Project:", project_path)
print("Data   :", paths["atd"].parent)

Project: E:\Hackathon\more-than-a-hack-2027
Data   : E:\Hackathon\more-than-a-hack-2027\data\tactical


In [16]:
TEAM_NAME = "BAJANG_TACTIC"   # GANTI bila nama tim di formulir pendaftaran berbeda
submission_csv = project_path / "submissions" / f"{TEAM_NAME}.csv"

import subprocess, shutil

# Otomatis mendeteksi apakah folder evaluation/ ada di root proyek atau di dalam official-repository/
if (project_path / "evaluation").exists():
    official_repo = project_path
else:
    official_repo = project_path / "official-repository" / "more-than-a-hack-2027"

def run_official(script, *args):
    script_path = official_repo / "evaluation" / script
    result = subprocess.run(
        [sys.executable, str(script_path), *map(str, args)],
        cwd=str(official_repo),
        capture_output=True,
        text=True
    )
    return (result.stdout + result.stderr).strip()

print("Folder evaluasi:", official_repo / "evaluation")

Folder evaluasi: E:\Hackathon\more-than-a-hack-2027\evaluation


In [17]:
pr.main(["--team", TEAM_NAME])

Submission: E:\Hackathon\more-than-a-hack-2027\submissions\BAJANG_TACTIC.csv
MD5       : 02eac74930755e82832be428b446eb28
Babak 1: frames_H=35197, frames_A=27885, frames_D=16385, frames_X=0, n_possessions=192, n_dead_stretches=54, n_direct_H_A_switches=137, mean_possession_s=13.1, median_possession_s=8.2, mean_dead_s=12.1
Babak 2: frames_H=32602, frames_A=24362, frames_D=22426, frames_X=0, n_possessions=171, n_dead_stretches=61, n_direct_H_A_switches=111, mean_possession_s=13.3, median_possession_s=8.6, mean_dead_s=14.7


In [18]:
print(run_official("validate_submission.py", submission_csv.resolve()))

OK    BAJANG_TACTIC.csv
      199319 rows, frame 0..199318

      label counts
        H    67799   34.0%
        A    52247   26.2%
        D    38811   19.5%
        X    40462   20.3%

      reports 363 possessions of mean 13.2 s
      (counted with each X as a gap; the scorer bridges abstentions inside a real
       possession, so its count can be lower than this one)

Valid. Note this checks the format only -- it says nothing about whether the labels are right.
Score it against the public ground truth with run_evaluation.py.


In [19]:
results_dir = official_repo / "evaluation" / "results"
shutil.copy2(submission_csv, results_dir / submission_csv.name)
print(run_official("run_evaluation.py"))
(results_dir / submission_csv.name).unlink(missing_ok=True)

scoreboard  gt\BAR-ATH_possession_gt_public.csv  199319 frames  7500 graded  21 gt segments
ground truth has 17 possessions of mean 14.2 s  <- compare the poss and dur columns
ranked on spell_f1_mean; the ~0.04 noise band was calibrated over a full match and this ground truth has only 17 possessions -- run --bootstrap 1000 for the real band
spellF1 = mean possession-spell F1 over IoU 0.3/0.4/0.5/0.6/0.7; transF1/lat_f at +/-25 frames

rank  submission                macroF1  spellF1   poss     dur  transF1  lat_f  shareE  frag
----  ------------------------  -------  -------  -----  ------  -------  -----  ------  ----
   1  BAJANG_TACTIC              0.6082   0.5500     15    19.6   0.2778   -6.0   0.002  0.81
   2  baseline_smooth125         0.5812   0.4242     16    18.5   0.1667   -7.7   0.052  0.81
   3  baseline_smooth50          0.6207   0.3256     26    10.8   0.0816   +4.0   0.048  1.43
   4  baseline_smooth25          0.6200   0.2868     36     7.6   0.2034   +8.2   0.056  1.

In [20]:
labels = pd.read_csv(submission_csv)["label"].astype(str).str.strip().str.upper().to_numpy()
graded = pm.get_graded_mask(len(labels))

print("Jumlah baris:", len(labels), "(harus 199319)")
print("Label unik  :", sorted(set(labels)))
print("X di frame dinilai   :", int(np.sum(labels[graded] == "X")), "(sebaiknya 0)")
print("Label non-X di luar babak:", int(np.sum(labels[~graded] != "X")), "(harus 0)")
for half, (s, e) in enumerate(pr.HALVES, start=1):
    print(f"Babak {half}:", ta.summarize_runs(labels, s, e))

Jumlah baris: 199319 (harus 199319)
Label unik  : ['A', 'D', 'H', 'X']
X di frame dinilai   : 0 (sebaiknya 0)
Label non-X di luar babak: 0 (harus 0)
Babak 1: {'frames_H': 35197, 'frames_A': 27885, 'frames_D': 16385, 'frames_X': 0, 'n_possessions': 192, 'n_dead_stretches': 54, 'n_direct_H_A_switches': 137, 'mean_possession_s': 13.142083333333334, 'median_possession_s': 8.2, 'mean_dead_s': 12.137037037037038}
Babak 2: {'frames_H': 32602, 'frames_A': 24362, 'frames_D': 22426, 'frames_X': 0, 'n_possessions': 171, 'n_dead_stretches': 61, 'n_direct_H_A_switches': 111, 'mean_possession_s': 13.324912280701751, 'median_possession_s': 8.6, 'mean_dead_s': 14.705573770491803}


In [21]:
import hashlib
first = hashlib.md5(submission_csv.read_bytes()).hexdigest()

check_csv = project_path / "experiments" / "repro_check.csv"
pr.main(["--team", TEAM_NAME, "--out", str(check_csv)])
second = hashlib.md5(check_csv.read_bytes()).hexdigest()
check_csv.unlink(missing_ok=True)

print("MD5 pertama:", first)
print("MD5 kedua  :", second)
print("IDENTIK" if first == second else "BERBEDA - periksa sumber ketidakpastian!")

Submission: E:\Hackathon\more-than-a-hack-2027\experiments\repro_check.csv
MD5       : 02eac74930755e82832be428b446eb28
Babak 1: frames_H=35197, frames_A=27885, frames_D=16385, frames_X=0, n_possessions=192, n_dead_stretches=54, n_direct_H_A_switches=137, mean_possession_s=13.1, median_possession_s=8.2, mean_dead_s=12.1
Babak 2: frames_H=32602, frames_A=24362, frames_D=22426, frames_X=0, n_possessions=171, n_dead_stretches=61, n_direct_H_A_switches=111, mean_possession_s=13.3, median_possession_s=8.6, mean_dead_s=14.7
MD5 pertama: 02eac74930755e82832be428b446eb28
MD5 kedua  : 02eac74930755e82832be428b446eb28
IDENTIK
